In [ ]:
import polars as pl
import time

# Python Polars Essential Training - Day 2

Welcome back! Today we'll cover advanced patterns that make Polars shine:
- **Time Series**: Resampling, rolling windows, and asof joins
- **Nested Data**: Working with Lists and Structs
- **Optimization**: Writing efficient Polars code
- **Custom Extensions**: When you need to plug in external libraries

Let's load our datasets first.

In [ ]:
sensors = pl.read_parquet("data/sensor_readings.parquet")
trades = pl.read_parquet("data/trades.parquet")
quotes = pl.read_parquet("data/quotes.parquet")
orders = pl.read_parquet("data/orders.parquet")

print(f"Sensors: {sensors.shape}, Trades: {trades.shape}, Quotes: {quotes.shape}, Orders: {orders.shape}")

## Time Series & Asof Joins

Polars has first-class support for time series data. Let's explore the temporal types and operations.

### Downsampling with `group_by_dynamic`

The sensor data is at minute-level granularity. Downsample it to **daily** data.

For each sensor, calculate:
- The mean value per day
- The max value per day

**Bonus**: Instead of manually `.alias()`-ing each column, use `.name.suffix()` to automatically add a suffix to aggregated column names.

See [DataFrame.group_by_dynamic](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.group_by_dynamic.html) · [Expr.name.suffix](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.name.suffix.html).

In [ ]:
# Your code here

### Rolling Windows

Calculate a **rolling mean** of the sensor values over a 1-hour window.

Filter to just `sensor_A` first and add a column `rolling_mean_1h` with the rolling mean.

See [Expr.rolling_mean_by](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.rolling_mean_by.html).

In [ ]:
# Your code here

### `join_asof`

We have `trades` (trade executions) and `quotes` (price updates) that are NOT synchronized.

For each trade, find the **last known price** from the quotes table using `join_asof`.

Steps:
1. Sort both DataFrames by their time columns
2. Use `join_asof` with `strategy="backward"` to find the most recent quote before each trade
3. The join should match on `symbol` as well

> **Note**: `join_asof` requires sorted input. Passing unsorted data raises an `InvalidOperationError`. Setting `check_sortedness=False` skips the check but silently produces wrong results, in this case mostly nulls. Always sort first. Or if your data is sorted, let Polars know by using `df.set_sorted("col")`.

See [DataFrame.join_asof](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.join_asof.html).

In [ ]:
# Your code here

### `join_where`

`join_where` lets you join on **inequality** predicates. Multiple predicates are AND-ed together like in `df.filter()`.

**Scenario**: You have acceptable threshold ranges for each sensor unit type. Find all readings that fall **outside** their safe range.

```python
thresholds = pl.DataFrame({
    "unit": ["°C", "hPa", "%RH", "lux"],
    "min_threshold": [18.0, 990.0, 30.0, 100.0],
    "max_threshold": [28.0, 1040.0, 70.0, 800.0],
})
```

1. Use `join_where` to find all readings **below** the minimum threshold and **above** the maximum threshold in the `sensors` dataset.
2. Group by `sensor_id` and show how many values are outside of these threshold per sensor.

**Hint**: When `unit` appears in both DataFrames, the right-side column is renamed `unit_right`.

See [DataFrame.join_where](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.join_where.html).

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Downsampling with group_by_dynamic</summary>

> ```python
> daily_sensors = sensors.group_by_dynamic(
>     "timestamp",
>     every="1d",
>     group_by="sensor_id",
> ).agg(
>     pl.col("value").mean().alias("mean_value"),
>     pl.col("value").max().alias("max_value"),
> )
> daily_sensors
> ```
> ```python
> # Bonus: .name.suffix() avoids repeating .alias() for every aggregation
> daily_sensors_alt = sensors.group_by_dynamic(
>     "timestamp",
>     every="1d",
>     group_by="sensor_id",
> ).agg(
>     pl.col("value").mean().name.suffix("_daily_mean"),
>     pl.col("value").max().name.suffix("_daily_max"),
> )
> daily_sensors_alt
> ```
</details>

<details>
<summary>Show Answer: Rolling Windows</summary>

> ```python
> sensor_a = sensors.filter(pl.col("sensor_id") == "sensor_A").sort("timestamp")
> 
> sensor_a_rolling = sensor_a.with_columns(
>     pl.col("value")
>     .rolling_mean_by("timestamp", window_size="1h")
>     .alias("rolling_mean_1h")
> )
> sensor_a_rolling.head(100)
> ```
</details>

<details>
<summary>Show Answer: join_asof</summary>

> ```python
> trades_sorted = trades.sort("trade_time")
> quotes_sorted = quotes.sort("quote_time")
> 
> trades_with_price = trades_sorted.join_asof(
>     quotes_sorted,
>     left_on="trade_time",
>     right_on="quote_time",
>     by="symbol",
>     strategy="backward",
> )
> trades_with_price.head(10)
> ```
</details>

<details>
<summary>Show Answer: join_where</summary>

> ```python
> thresholds = pl.DataFrame({
>     "unit": ["°C", "hPa", "%RH", "lux"],
>     "min_threshold": [18.0, 990.0, 30.0, 100.0],
>     "max_threshold": [28.0, 1040.0, 70.0, 800.0],
> })
> 
> out_of_range = sensors.join_where(
>     thresholds,
>     pl.col("unit") == pl.col("unit_right"),
>     (pl.col("value") < pl.col("min_threshold")) | (pl.col("value") > pl.col("max_threshold")),
> )
> print(f"Total out-of-range readings: {len(out_of_range)}")
> out_of_range.group_by("sensor_id").len().sort("sensor_id")
> ```
</details>

#### End of section, wait here


## Handling Nested Data (Lists & Structs)

This is where many users fall back to Python loops. Polars handles nested data natively!

### Reading & Exploring Nested Data

Real-world data often comes as JSON/NDJSON with nested objects. Polars maps these to **Struct** and **List** types.

1. Read `data/github_events.ndjson` using `pl.read_ndjson()`
2. Examine the schema - notice how nested JSON objects become Structs
3. How many events are there? What event types exist?

**Tip**: NDJSON (newline-delimited JSON) is preferred for large datasets because it can be streamed line-by-line.

See [pl.read_ndjson](https://docs.pola.rs/api/python/dev/reference/api/polars.read_ndjson.html).

In [ ]:
# Your code here

### Unnesting Structs & Accessing Fields

Using the GitHub events data:

1. Use `unnest()` to flatten the `repo` and `actor` structs into separate columns
2. Alternatively, use `.struct.field("name")` to extract just the repo name without unnesting
3. Create a summary showing: event type, repo name, and actor login

See [DataFrame.unnest](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.unnest.html) and [Expr.struct.field](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.struct.field.html).

In [ ]:
# Your code here

### Working with Lists: Filter, Explode & More

Using the `orders` DataFrame (loaded from `data/orders.parquet` in cell 2 above):
1. Filter to orders where `tags` contains `"premium"` (see [Expr.list.contains](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.list.contains.html))
2. Add a column `n_tags` with the tag count (see [Expr.list.len](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.list.len.html))

Then with the `events` DataFrame (loaded from `data/github_events.ndjson`):

3. Filter to `push` events and extract the `commits` list from the payload
4. Use `explode()` to create one row per commit (see [DataFrame.explode](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.explode.html))
5. Extract `sha` and `message` from each commit struct


In [ ]:
# Your code here

### Grouping into Lists with `implode`

Using the `events` DataFrame (from `data/github_events.ndjson`, loaded in the answer above), create a per-repo summary of push events containing:
- Total number of commits across all push events for that repo
- List of unique committer logins

**Hint**: Use `group_by(...).agg(...)`. Inside `agg`, aggregation expressions already produce a list per group, so you do not need an extra `.implode()`.

See [Expr.list.unique](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.list.unique.html),
[GroupBy.agg user guide](https://docs.pola.rs/user-guide/expressions/aggregation/).


In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Reading & Exploring Nested Data</summary>

> ```python
> events = pl.read_ndjson("data/github_events.ndjson")
> print("Schema:")
> print(events.schema)
> print(f"\nEvents: {len(events)}")
> print(f"\nEvent types: {events['type'].unique().to_list()}")
> events.head()
> ```
</details>

<details>
<summary>Show Answer: Unnesting Structs & Accessing Fields</summary>

> ```python
> events_flat = events.unnest("repo").unnest("actor")
> print("Columns after unnest:", events_flat.columns)
>
> # Alternative: extract fields directly
> events.select(
>     "type",
>     pl.col("repo").struct.field("name").alias("repo_name"),
>     pl.col("actor").struct.field("login").alias("actor_login"),
> )
> ```
</details>

<details>
<summary>Show Answer: Working with Lists: Filter, Explode & More</summary>

> ```python
> premium_orders = orders.filter(pl.col("tags").list.contains("premium"))
> print(f"Premium orders: {len(premium_orders)}")
>
> orders_with_count = orders.with_columns(
>     pl.col("tags").list.len().alias("n_tags")
> )
> ```
</details>

<details>
<summary>Show Answer: Grouping into Lists with implode</summary>

> ```python
> # Explode commits first so each commit is its own row
> push_commits = (
>     events
>     .filter(pl.col("type") == "push")
>     .select(
>         pl.col("repo").struct.field("name").alias("repo_name"),
>         pl.col("actor").struct.field("login").alias("actor_login"),
>         pl.col("payload").struct.field("commits").alias("commits"),
>     )
>     .explode("commits")
>     .with_columns(
>         pl.col("commits").struct.field("sha").alias("sha"),
>     )
>     .drop("commits")
> )
>
> # Now group by repo and aggregate: total commits, unique committers list
> repo_summary = (
>     push_commits
>     .group_by("repo_name")
>     .agg(
>         pl.len().alias("total_commits"),
>         pl.col("actor_login").unique().alias("committers"),
>     )
> )
> repo_summary
> ```
</details>


#### End of section, wait here


## Internals & "The Polars Way"

Now let's learn how to write efficient Polars code and understand what happens under the hood.

### Visualizing Query Plans with `show_graph()`

> **Note**: If `show_graph()` errors with "graphviz not found", see the install cell at the top of Day 1.

Use `show_graph()` to visualize the query plan for the following lazy query. It joins sensor readings
with a small thresholds table, then filters and aggregates — giving the optimizer something real to chew on:

```python
thresholds = pl.LazyFrame({
    "sensor_id": ["sensor_A", "sensor_B", "sensor_C", "sensor_D"],
    "warn_above": [22.0, 18.0, 30.0, 800.0],
})

query = (
    pl.scan_parquet("data/sensor_readings.parquet")
    .join(thresholds, on="sensor_id", how="inner")
    .filter(pl.col("value") > pl.col("warn_above"))
    .group_by("sensor_id")
    .agg(
        pl.len().alias("n_warnings"),
        pl.col("value").max().alias("peak"),
    )
)
```

Compare three views of the same query:
1. `query.show_graph()`: optimized IR (the default)
2. `query.show_graph(optimized=False)`: unoptimized IR, your query as written
3. `query.show_graph(plan_stage="physical")`: physical execution plan

Questions:
- Is **Predicate Pushdown** happening? (Filters pushed into the scan)
- Is **Projection Pushdown** happening? (Only `sensor_id`, `value` read from disk)
- Is join-side filter pushdown visible? (Does the optimizer push any filter before the join?)
- What differences do you see between the optimized IR and the physical plan?

See [LazyFrame.show_graph](https://docs.pola.rs/api/python/dev/reference/lazyframe/api/polars.LazyFrame.show_graph.html).

In [ ]:
# Your code here

### Streaming for Large Data

When working with data larger than RAM, Polars can process it in chunks using the streaming engine.

The query below stacks the sensor data 50 times (simulating a larger dataset) and runs a join + filter + aggregation.
Compare two collect strategies:
1. `query.collect()`: loads all data into memory at once
2. `query.collect(engine="streaming")`: processes in morsels, lower peak memory and faster.

Use `benchmark()` to time both. On larger data, streaming amortizes its per-chunk overhead and wins.

```python
thresholds = pl.LazyFrame({
    "sensor_id": ["sensor_A", "sensor_B", "sensor_C", "sensor_D"],
    "warn_above": [22.0, 18.0, 30.0, 800.0],
})

query = (
    pl.concat([pl.scan_parquet("data/sensor_readings.parquet")] * 50)
    .join(thresholds, on="sensor_id", how="inner")
    .filter(pl.col("value") > pl.col("warn_above"))
    .group_by("sensor_id")
    .agg(
        pl.len().alias("n_warnings"),
        pl.col("value").mean().alias("mean_warn"),
        pl.col("value").max().alias("peak"),
    )
)
```

**Bonus**: Instead of passing `engine="streaming"` to every `collect()` call, set it globally with
`pl.Config.set_engine_affinity("streaming")`. The streaming engine will become the default in a future
Polars release, so this is a good way to future-proof your code.

See [LazyFrame.collect](https://docs.pola.rs/api/python/dev/reference/lazyframe/api/polars.LazyFrame.collect.html),
[pl.Config.set_engine_affinity](https://docs.pola.rs/api/python/dev/reference/api/polars.Config.set_engine_affinity.html).


In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Visualizing Query Plans</summary>

> ```python
> thresholds = pl.LazyFrame({
>     "sensor_id": ["sensor_A", "sensor_B", "sensor_C", "sensor_D"],
>     "warn_above": [22.0, 18.0, 30.0, 800.0],
> })
>
> query = (
>     pl.scan_parquet("data/sensor_readings.parquet")
>     .join(thresholds, on="sensor_id", how="inner")
>     .filter(pl.col("value") > pl.col("warn_above"))
>     .group_by("sensor_id")
>     .agg(
>         pl.len().alias("n_warnings"),
>         pl.col("value").max().alias("peak"),
>     )
> )
>
> # 1. Optimized IR (default)
> print("Optimized IR:")
> query.show_graph()
> ```
> ```python
> # 2. Unoptimized IR
> print("Unoptimized IR:")
> query.show_graph(optimized=False)
> ```
> ```python
> # 3. Physical execution plan
> print("Physical plan:")
> query.show_graph(plan_stage="physical")
> ```
</details>

<details>
<summary>Show Answer: Streaming for Large Data</summary>

> ```python
> thresholds = pl.LazyFrame({
>     "sensor_id": ["sensor_A", "sensor_B", "sensor_C", "sensor_D"],
>     "warn_above": [22.0, 18.0, 30.0, 800.0],
> })
>
> query = (
>     pl.concat([pl.scan_parquet("data/sensor_readings.parquet")] * 50)
>     .join(thresholds, on="sensor_id", how="inner")
>     .filter(pl.col("value") > pl.col("warn_above"))
>     .group_by("sensor_id")
>     .agg(
>         pl.len().alias("n_warnings"),
>         pl.col("value").mean().alias("mean_warn"),
>         pl.col("value").max().alias("peak"),
>     )
> )
>
> regular_time = benchmark(lambda: query.collect())
> streaming_time = benchmark(lambda: query.collect(engine="streaming"))
>
> print(f"Regular:   {regular_time:.4f}s")
> print(f"Streaming: {streaming_time:.4f}s")
> print(f"Streaming is {regular_time / streaming_time:.1f}x faster!")
> ```
> ```python
> # Bonus: set streaming as global default
> pl.Config.set_engine_affinity("streaming")
> result = query.collect()
> print(result)
> ```
</details>


#### End of section, wait here


## Extending Polars with Custom Functionality

Sometimes you need functionality that doesn't exist in Polars. For these cases, you can use:
- `map_elements`: Apply a Python function to each element (row-by-row, slow but flexible)
- `map_batches`: Apply a function to entire columns as Series (vectorized, faster)

**Warning**: These should be your last resort! Native expressions are always faster.

In [ ]:
def benchmark(func, runs=5):
    # runs=5, return min: system noise only ever makes a run slower, min best estimates intrinsic speed
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        func()
        times.append(time.perf_counter() - start)
    return min(times)


In [ ]:
# Your code here

### The Cost of Leaving Rust: A Benchmark

Now let's compare a pattern for string replacements. Compare these two options:

- `map_elements(lambda x: x.replace("sensor_", "S_"))` on the `sensor_id` column
- vs. native `pl.col("sensor_id").str.replace("sensor_", "S_")`

Print the execution time and speedup factor for each using the `benchmark` function.

See [Expr.map_elements](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.map_elements.html), [Expr.str.replace](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.str.replace.html).


In [ ]:
# Your code here

### Sentiment Analysis with TextBlob

This is a real-world example of extending Polars with external libraries.

Use [TextBlob](https://textblob.readthedocs.io/) to add sentiment scores to the reviews.

**Steps**:
1. Create a function that takes a text string and returns sentiment polarity (-1 to 1)
2. Apply it using `map_elements`
3. Categorize as 'positive', 'negative', or 'neutral'

**Note**: TextBlob is installed in your environment and can be imported lik `from textblob import TextBlob`

```python
reviews = pl.DataFrame({
    "text": [
        "Absolutely fantastic, exceeded every expectation!",
        "I don't really care, it was meh",
        "Terrible experience, would not recommend at all",
    ]
})
```

See [Expr.map_elements](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.map_elements.html).

In [ ]:
# Your code here

### Using `map_batches` for Vectorized Operations

`map_batches` receives the entire column as a Series, which is more efficient for operations that can work on arrays.

Create a function using `map_batches` that:
1. Applies the function to the `value` column from `sensors` (`data/sensor_readings.parquet`, loaded in cell 2) filtered to `sensor_id == "sensor_A"`
2. The function receives the column as a Series and implements custom row-wise normalization: `(value - column_min) / (column_max - column_min)`
3. Returns a Series with the normalized values

See [Expr.map_batches](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.map_batches.html).


In [ ]:
# Your code here

### Multi-Column UDFs: Using Structs

What if your Python function needs **multiple columns** as input? You can't pass them directly to `map_elements`.

The idiomatic Polars pattern is:
1. Combine columns into a **struct** using `pl.struct([col1, col2, ...])`
2. Apply your function to the struct column
3. Your function receives each row as a dict-like object

**Exercise**: Create a `calculate_risk` function using `value` and `unit` from the `sensors` DataFrame (`data/sensor_readings.parquet`, loaded in cell 2):
- Temperature (`°C`): `value / 25`
- Pressure (`hPa`): `abs(value - 1013) / 50` (deviation from sea level)
- Humidity (`%RH`): `abs(value - 50) / 30` (deviation from ideal 50%)
- Otherwise: `0.5`

```python
def calculate_risk(row: dict) -> float:
    value = row["value"]
    unit = row["unit"]
    # Your logic here
    pass

sensors.with_columns(
    pl.struct(["value", "unit"])
    .map_elements(calculate_risk, return_dtype=pl.Float64)
    .alias("risk_score")
)
```

**Bonus**: Rewrite using native `pl.when().then()` chains.

See [pl.struct](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.struct.html).


In [ ]:
# Your code here

### Testing Your Transformations

When you create custom functions that transform DataFrames, you should test them!

Polars provides `pl.testing.assert_frame_equal()` to verify your results.

**Exercise**: Create a function `add_sentiment_category` that:
1. Takes a DataFrame with a `sentiment_score` column
2. Adds a `category` column: "positive" (>0.1), "negative" (<-0.1), or "neutral"

Then test it using the expected output below:

See [polars.testing.assert_frame_equal](https://docs.pola.rs/api/python/dev/reference/api/polars.testing.assert_frame_equal.html).


In [ ]:
import polars.testing as pl_test

# Your function
def add_sentiment_category(df: pl.DataFrame) -> pl.DataFrame:
    # Your code here
    pass

# Test case
input_df = pl.DataFrame({"sentiment_score": [0.5, -0.3, 0.0]})
expected = pl.from_repr("""
┌─────────────────┬──────────┐
│ sentiment_score ┆ category │
│ ---             ┆ ---      │
│ f64             ┆ str      │
╞═════════════════╪══════════╡
│ 0.5             ┆ positive │
│ -0.3            ┆ negative │
│ 0.0             ┆ neutral  │
└─────────────────┴──────────┘
""")

# This will raise AssertionError if they don't match
pl_test.assert_frame_equal(add_sentiment_category(input_df), expected)
print("Test passed!")

### Answers

<details>
<summary>Show Answer: The Cost of Leaving Rust: A Benchmark</summary>

> ```python
> def slow_replace():
>     return sensors.with_columns(
>         pl.col("sensor_id").map_elements(
>             lambda x: x.replace("sensor_", "S_"), return_dtype=pl.Utf8
>         ).alias("sensor_id_short")
>     )
>
> def fast_replace():
>     return sensors.with_columns(
>         pl.col("sensor_id").str.replace("sensor_", "S_").alias("sensor_id_short")
>     )
>
> slow = benchmark(slow_replace)
> fast = benchmark(fast_replace)
> print(f"map_elements: {slow:.4f}s")
> print(f"str.replace:  {fast:.4f}s")
> print(f"Speedup: {slow / fast:.1f}x")
> ```
</details>

<details>
<summary>Show Answer: Sentiment Analysis with TextBlob</summary>

> ```python
> from textblob import TextBlob
>
> reviews = pl.DataFrame({
>     "text": [
>         "Absolutely fantastic, exceeded every expectation!",
>         "I don't really care, it was meh",
>         "Terrible experience, would not recommend at all",
>     ]
> })
>
> def get_sentiment(text: str) -> float:
>     return TextBlob(text).sentiment.polarity
>
> reviews_with_sentiment = reviews.with_columns(
>     pl.col("text").map_elements(get_sentiment, return_dtype=pl.Float64).alias("sentiment_score")
> ).with_columns(
>     pl.when(pl.col("sentiment_score") > 0.1).then(pl.lit("positive"))
>     .when(pl.col("sentiment_score") < -0.1).then(pl.lit("negative"))
>     .otherwise(pl.lit("neutral"))
>     .alias("sentiment_label")
> )
> reviews_with_sentiment
> ```
</details>

<details>
<summary>Show Answer: map_batches</summary>

> ```python
> sensor_a = sensors.filter(pl.col("sensor_id") == "sensor_A")
>
> def normalize(s: pl.Series) -> pl.Series:
>     return (s - s.min()) / (s.max() - s.min())
>
> sensor_a.with_columns(
>     pl.col("value").map_batches(normalize).alias("value_normalized")
> )
> ```
</details>

<details>
<summary>Show Answer: Multi-Column UDFs</summary>

> ```python
> def calculate_risk(row: dict) -> float:
>     value = row["value"]
>     unit = row["unit"]
>     if unit == "°C":
>         return value / 25
>     elif unit == "hPa":
>         return abs(value - 1013) / 50
>     elif unit == "%RH":
>         return abs(value - 50) / 30
>     return 0.5
>
> sensors.with_columns(
>     pl.struct(["value", "unit"])
>     .map_elements(calculate_risk, return_dtype=pl.Float64)
>     .alias("risk_score")
> )
> ```
</details>

<details>
<summary>Show Answer: Testing Your Transformations</summary>

> ```python
> import polars.testing as pl_test
>
> def add_sentiment_category(df: pl.DataFrame) -> pl.DataFrame:
>     return df.with_columns(
>         pl.when(pl.col("sentiment_score") > 0.1).then(pl.lit("positive"))
>         .when(pl.col("sentiment_score") < -0.1).then(pl.lit("negative"))
>         .otherwise(pl.lit("neutral"))
>         .alias("category")
>     )
>
> input_df = pl.DataFrame({"sentiment_score": [0.5, -0.3, 0.0]})
> expected = pl.DataFrame({
>     "sentiment_score": [0.5, -0.3, 0.0],
>     "category": ["positive", "negative", "neutral"]
> })
>
> pl_test.assert_frame_equal(add_sentiment_category(input_df), expected)
> print("Test passed!")
> ```
</details>


#### End of section, wait here


## Afternoon Capstone: IoT Anomaly Detection

**Scenario**: You have sensor readings from an IoT system in `data/sensor_readings.parquet` (loaded as `sensors` in cell 2). Your task is to build an anomaly detection pipeline.

### Requirements:
1. Use `scan_parquet` with streaming to load the sensor data
2. Calculate a rolling standard deviation (1-hour window) for each sensor
3. Flag readings as "anomaly" where the value deviates more than 2 standard deviations from the rolling mean
4. Group results by sensor and create a nested summary with:
   - Total readings
   - Number of anomalies
   - List of anomaly timestamps


In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: IoT Anomaly Detection Pipeline</summary>

> ```python
> 
> # Step 1: Load with lazy API and sort by timestamp
> sensor_data = (
>     pl.scan_parquet("data/sensor_readings.parquet")
>     .sort("timestamp")
>     .collect()
> )
> 
> # Step 2 & 3: Calculate rolling stats and flag anomalies
> # Process each sensor group separately for correct rolling windows
> anomaly_detection = sensor_data.with_columns(
>     pl.col("value")
>     .rolling_mean_by("timestamp", window_size="1h")
>     .over("sensor_id")
>     .alias("rolling_mean"),
>     pl.col("value")
>     .rolling_std_by("timestamp", window_size="1h")
>     .over("sensor_id")
>     .alias("rolling_std"),
> ).with_columns(
>     # Flag anomalies: value > 2 std from rolling mean
>     pl.when(
>         (pl.col("value") - pl.col("rolling_mean")).abs() 
>         > (2 * pl.col("rolling_std"))
>     )
>     .then(pl.lit(True))
>     .otherwise(pl.lit(False))
>     .alias("is_anomaly")
> )
> 
> print(f"Total readings: {len(anomaly_detection)}")
> print(f"Anomalies detected: {anomaly_detection.filter(pl.col('is_anomaly')).height}")
> ```
</details>

<details>
<summary>Show Answer: Creating Nested Summary</summary>

> ```python
> # Step 4: Create nested summary
> summary = anomaly_detection.group_by("sensor_id").agg(
>     pl.len().alias("total_readings"),
>     pl.col("is_anomaly").sum().alias("anomaly_count"),
>     # List of anomaly timestamps
>     pl.col("timestamp").filter(pl.col("is_anomaly")).implode().alias("anomaly_timestamps"),
>     # Also get the anomaly values for context
>     pl.col("value").filter(pl.col("is_anomaly")).implode().alias("anomaly_values"),
> )
> 
> summary
> ```
</details>

<details>
<summary>Show Answer: Bonus: JSON Export</summary>

> ```python
> # Bonus: Create a struct for JSON-like export
> export_ready = summary.with_columns(
>     pl.struct([
>         "total_readings",
>         "anomaly_count", 
>         "anomaly_timestamps",
>         "anomaly_values"
>     ]).alias("sensor_report")
> ).select("sensor_id", "sensor_report")
> 
> export_ready
> ```
</details>


#### End of section, wait here


## Bonus: Visualization

You can use visualization libraries like `altair` or `hvPlot` to visualize the detected anomalies.
Polars integrates well with Altair/Vega-Lite.

Polars also has a built-in lightweight plotting API via `DataFrame.plot` (backed by hvPlot) — useful for
quick exploration without importing Altair. See [DataFrame.plot](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.plot.html).


In [ ]:
import altair as alt

try:
    # Note: anomaly_detection comes from your capstone solution above
    plot_data = anomaly_detection.filter(pl.col("sensor_id") == "sensor_A").sample(5000)

    base = alt.Chart(plot_data).encode(x="timestamp")
    line = base.mark_line().encode(y="value")
    points = base.mark_circle(color="red", size=60).encode(y="value").transform_filter(alt.datum.is_anomaly)
    line + points
except NameError:
    print("Complete the capstone exercise above to visualize results.")
